In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# 1. Load Data
wdi = pd.read_parquet('raw_wdi_wdi_data.parquet')
imf = pd.read_parquet('raw_imf_imf_data.parquet')
meta = pd.read_parquet('raw_imf_metadata_imf_metadata.parquet')
hosts_df = pd.read_csv('raw_fifa_wc_hosts.csv')

# Combine and setup metadata lookup
econ_data = pd.concat([wdi, imf], ignore_index=True)
meta_lookup = dict(zip(meta.indicator_code, meta.indicator_label))

print("Data loaded successfully.")

Data loaded successfully.


In [ ]:
# 1. Align Features
latest_year = econ_data['year'].max()
cand_df = econ_data[econ_data['year'] == latest_year].pivot(index='iso3', columns='indicator_code', values='value')

host_profiles = []
for _, row in hosts_df.iterrows():
    snap = econ_data[(econ_data['iso3'] == row['iso3']) & (econ_data['year'] == row['year'])]
    if not snap.empty:
        p = snap.pivot(index='iso3', columns='indicator_code', values='value')
        host_profiles.append(p)

train_full = pd.concat(host_profiles)
common = train_full.columns.intersection(cand_df.columns)

# 2. Log-Normalization (Critical for managing the gap between Trillions and Billions)
X_train_raw = train_full[common]
X_cand_raw = cand_df[common]

X_train_log = np.sign(X_train_raw) * np.log1p(np.abs(X_train_raw))
X_cand_log = np.sign(X_cand_raw) * np.log1p(np.abs(X_cand_raw))

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(imputer.fit_transform(X_train_log))
X_cand_scaled = scaler.transform(imputer.transform(X_cand_log))

# 3. PCA with Strategic Calibration
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_cand_pca = pca.transform(X_cand_scaled)

# --- THE CALIBRATION FIX ---
# We force the model to acknowledge that USA/CHN/FRA represent the 'Positive' direction
# If 'USA' is to the left of the 'ZWE' (Zimbabwe), we flip the X-axis.
if 'USA' in cand_df.index and 'ZWE' in cand_df.index:
    usa_pos = X_cand_pca[cand_df.index.get_loc('USA'), 0]
    zwe_pos = X_cand_pca[cand_df.index.get_loc('ZWE'), 0]
    if usa_pos < zwe_pos:
        X_train_pca[:, 0] *= -1
        X_cand_pca[:, 0] *= -1
        print("Calibrating orientation: High-capacity countries moved to the 'Elite' side.")
    else:
        print("Orientation confirmed: Economic scale is correctly mapped.")

# 4. Train OCSVM
ocsvm = OneClassSVM(kernel='rbf', gamma='auto', nu=0.05).fit(X_train_scaled)

Calibrating orientation: High-capacity countries moved to the 'Elite' side.


In [ ]:
def evaluate_bid_calibrated(iso3):
    if iso3 not in cand_df.index:
        return f"Error: {iso3} not found."

    idx = cand_df.index.get_loc(iso3)
    vec_scaled = X_cand_scaled[idx].reshape(1, -1)
    vec_pca = X_cand_pca[idx]

    # 1. Benchmarking against the Host Centroid
    host_center = X_train_pca.mean(axis=0)
    vector = vec_pca - host_center

    # Power Score: Scale (PC1) + Stability (PC2)
    power_score = (vector[0] * 1.5) + (vector[1] * 1.0)

    # 2. Score Normalization (Average = 50)
    all_host_raw = []
    for h_pca in X_train_pca:
        v = h_pca - host_center
        all_host_raw.append((v[0] * 1.5) + (v[1] * 1.0))

    all_host_raw = np.array(all_host_raw)

    # Linear Interpolation: Min Host = 20, Average Host = 50, Max Host = 95
    # This prevents 'Outlier Blindness' from USA/CHN
    viability_idx = np.interp(power_score,
                             [all_host_raw.min(), np.mean(all_host_raw), all_host_raw.max()],
                             [20, 50, 95])

    # 3. Identity Check (SVM)
    is_inlier = ocsvm.predict(vec_scaled)[0]

    # 4. Status Logic
    if is_inlier == -1 and vector[0] > 0:
        status, code = "ELITE CANDIDATE", "🟢"
    elif is_inlier == 1:
        status, code = "STANDARD VIABLE", "🔵"
    else:
        status, code = "HIGH RISK / DEVELOPING", "🔴"

    print(f"--- WORLD CUP BID REPORT: {iso3} ---")
    print(f"Status: {code} {status}")
    print(f"Viability Index: {viability_idx:.1f} / 100")
    print("-" * 45)

# Final Test Run
for t in ['USA', 'FRA', 'MAR', 'IND', 'ZWE']:
    evaluate_bid_calibrated(t)
    print("\n")

--- WORLD CUP BID REPORT: USA ---
Status: 🟢 ELITE CANDIDATE
Viability Index: 95.0 / 100
---------------------------------------------


--- WORLD CUP BID REPORT: FRA ---
Status: 🟢 ELITE CANDIDATE
Viability Index: 86.5 / 100
---------------------------------------------


--- WORLD CUP BID REPORT: MAR ---
Status: 🔵 STANDARD VIABLE
Viability Index: 43.6 / 100
---------------------------------------------


--- WORLD CUP BID REPORT: IND ---
Status: 🟢 ELITE CANDIDATE
Viability Index: 64.5 / 100
---------------------------------------------


--- WORLD CUP BID REPORT: ZWE ---
Status: 🔴 HIGH RISK / DEVELOPING
Viability Index: 20.0 / 100
---------------------------------------------


